# 18 — What no classical-data embedding can buy

We made the effect large in `17`. Now we ask the sharpest question of the whole capstone: is there *any* embedding — even one that **entangles** the two systems — that lets a quantum-optimal-transport measure detect coupling structure **no** classical statistic can? The honest answer, demonstrated here rather than asserted, is **no** — and seeing *why* turns the capstone's caveat into a precise structural boundary, and tells us exactly where a genuinely quantum advantage *would* have to come from.

**Prerequisites:** `16_capstone_synthesis` (the "no quantum magic" caveat), `17_embedding_design_amplifies_the_effect` (the embedding as a dial), `14_richer_embedding_applied` (the qutrit and its moment channels), `18`… in spirit also `01/18_entanglement`.

**What you'll be able to do**
- State the **structural bound**: for any *deterministic* embedding of *classical* phase data, every element of $\rho_{AB}$ is a classical (cross-)moment, so a same-order classical statistic matches any QOT measure.
- Run the decisive test: two ensembles matched on the first *and* second circular moments, differing only in the **third** — and watch a too-low-order embedding stay blind even when made genuinely **entangling**.
- Name precisely **when the bound would break** (intrinsically quantum data, or a quantum measurement/channel) — and why that is out of reach for EEG/MEG, including OPM-MEG.
- Place the whole programme correctly: a **quantum-inspired** method whose value is a unified geometric framework, not a detection advantage over classical statistics.

In `16` we were careful not to overclaim: a classical second-moment statistic could also separate the capstone's ensembles. That was stated as an observation. Here we promote it to a theorem-shaped statement and *stress-test* it — including with the one tool that might seem to break it, genuine entanglement. This is the honest frontier of the quantum-inspired program, found by experiment.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qot_course import viz
from qot_course.colors import COLORS
from qot_course.quantum_ot.capstone import coupling_qmi
from qot_course.quantum_ot.embeddings import (
    multifreq_state,
    joint_density_from_states,
    entangling_joint_density,
    controlled_shift_unitary,
)
from qot_course.quantum.composite import entanglement_entropy

np.random.seed(42)
viz.use_course_style()

N = 150_000

## 1. Why the bound is structural

Every embedding in this course turns *classical numbers* — the phases $\theta_A(t), \theta_B(t)$ — into a state by a *fixed, deterministic* rule, and then averages:
$$ \rho_{AB} = \mathbb{E}_t\bigl[\,|\psi_A(t)\rangle\langle\psi_A(t)| \otimes |\psi_B(t)\rangle\langle\psi_B(t)|\,\bigr]. $$
Look at any single matrix element. It is an **empirical average of a deterministic function of $(\theta_A, \theta_B)$** — for the multi-frequency embedding, an average of $e^{i(h\theta_A - h'\theta_B)}$, i.e. a **circular (cross-)moment** of the classical phases. So $\rho_{AB}$ is *built out of* classical moments, and any function of it — quantum mutual information, the Bures distance, a transport cost — is a function of those same classical moments.

The consequence is immediate and unavoidable: **a classical analyst with access to the same moments has everything $\rho_{AB}$ has.** A QOT measure cannot detect a difference that the matched set of classical moments does not already contain. This is not a fact about *our* embeddings — it is a fact about deterministically embedding classical data. The question is whether anything escapes it. The most promising candidate is entanglement, so we test exactly that.

## 2. The decisive test: matched on moments 1 and 2, differing only in moment 3

We build two ensembles that agree on the first **and** second circular moments of the phase difference, and differ **only** in the third moment $\langle e^{3i\delta}\rangle$. Then we ask four measures whether they can tell the ensembles apart:

- the **classical** third-harmonic statistic $|\langle e^{3i\delta}\rangle|$ — the fair baseline;
- the **qutrit** QMI with harmonics $(1,2)$ — which has *no channel* for the third moment;
- the **qudit** QMI with harmonics $(1,2,3)$ — which does;
- the **entangling** qutrit QMI with harmonics $(1,2)$ — the low-order embedding, but run through a genuine entangler.

The bound predicts: the qutrit $(1,2)$ is blind; the qudit $(1,2,3)$ sees it, *but so does the classical statistic*; and entangling the qutrit $(1,2)$ **does not** conjure access to the third moment.

In [ ]:
# Sample delta ~ 1 + 2 a1 cos d + 2 a2 cos 2d + 2 a3 cos 3d (rejection). A teaching
# one-off: the src sampler controls two harmonics; here we need three to MATCH moments 1,2
# and part only moment 3.
def sample_three_harmonic(n, a1, a2, a3, seed):
    rng = np.random.default_rng(seed)
    peak = 1 + 2 * (abs(a1) + abs(a2) + abs(a3))
    out = np.empty(n); filled = 0
    while filled < n:
        cand = rng.uniform(0, 2 * np.pi, n)
        dens = 1 + 2 * a1 * np.cos(cand) + 2 * a2 * np.cos(2 * cand) + 2 * a3 * np.cos(3 * cand)
        keep = cand[rng.uniform(0, peak, n) < dens]
        take = min(keep.size, n - filled)
        out[filled:filled + take] = keep[:take]; filled += take
    return out

rng = np.random.default_rng(111)
theta_a = rng.uniform(0, 2 * np.pi, N)
delta_lo = sample_three_harmonic(N, 0.4, 0.2, 0.0, seed=1)   # a3 = 0
delta_hi = sample_three_harmonic(N, 0.4, 0.2, 0.3, seed=2)   # a3 = 0.3, with a1, a2 matched
tb_lo = np.mod(theta_a - delta_lo, 2 * np.pi)
tb_hi = np.mod(theta_a - delta_hi, 2 * np.pi)

def plv_k(ta, tb, k):
    return abs(np.mean(np.exp(1j * k * (ta - tb))))

print("classical circular-moment gaps (|<e^ikd>| high - low):")
for k in (1, 2, 3):
    tag = "matched" if k < 3 else "the only classical difference (a3)"
    print(f"  k={k}:  {abs(plv_k(theta_a, tb_lo, k) - plv_k(theta_a, tb_hi, k)):.4f}   <- {tag}")

In [ ]:
def qmi_plain(ta, tb, harmonics):
    d = 1 + len(harmonics)
    rho = joint_density_from_states(multifreq_state(ta, harmonics), multifreq_state(tb, harmonics))
    return coupling_qmi(rho, dims=(d, d))

# Entangling embedding on harmonics (1,2): apply a fixed controlled-shift before averaging.
U = controlled_shift_unitary(3)
def qmi_entangled(ta, tb):
    rho = entangling_joint_density(multifreq_state(ta, (1, 2)), multifreq_state(tb, (1, 2)), U)
    return coupling_qmi(rho, dims=(3, 3)), rho

gap_qutrit = abs(qmi_plain(theta_a, tb_lo, (1, 2)) - qmi_plain(theta_a, tb_hi, (1, 2)))
gap_qudit = abs(qmi_plain(theta_a, tb_lo, (1, 2, 3)) - qmi_plain(theta_a, tb_hi, (1, 2, 3)))
q_lo, rho_e = qmi_entangled(theta_a, tb_lo)
q_hi, _ = qmi_entangled(theta_a, tb_hi)
gap_entangled = abs(q_lo - q_hi)
ent = entanglement_entropy(rho_e, dims=[3, 3])
gap_classical = abs(plv_k(theta_a, tb_lo, 3) - plv_k(theta_a, tb_hi, 3))

print(f"classical 3rd-harmonic statistic   gap = {gap_classical:.4f}   (sees a3)")
print(f"QMI qutrit  harmonics=(1,2)        gap = {gap_qutrit:.5f}   (BLIND to a3)")
print(f"QMI qudit   harmonics=(1,2,3)      gap = {gap_qudit:.5f}   (sees a3 -- so does the classical stat)")
print(f"QMI entangling qutrit (1,2)        gap = {gap_entangled:.5f}   (STILL blind, despite entanglement)")
print(f"  entanglement entropy of the entangled rho_AB = {ent:.3f} bits  (>0: genuinely entangled)")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.8))
labels = ["classical\n3rd-harmonic", "QMI qutrit\n(1,2)", "QMI qudit\n(1,2,3)", "QMI entangling\nqutrit (1,2)"]
vals = [gap_classical, gap_qutrit, gap_qudit, gap_entangled]
colors = [COLORS["positive"], COLORS["muted"], COLORS["quantum"], COLORS["highlight"]]
x = np.arange(len(labels))
ax.bar(x, vals, color=colors, edgecolor="white", linewidth=1.2, zorder=3, log=True)
for xi, v in zip(x, vals):
    ax.annotate(f"{v:.4f}", (xi, v), xytext=(0, 4), textcoords="offset points",
                ha="center", va="bottom", fontsize=9.5, color=COLORS["text"])
ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=9.5)
ax.set_ylabel("gap between the two ensembles  (log scale)")
ax.set_title("Only measures with a 3rd-moment channel separate them — entangling a (1,2) embedding does not", fontsize=11, pad=10)
ax.grid(axis="x", visible=False)
fig.tight_layout()
plt.show()

**Read the figure.** Four bars, log scale, for two ensembles that differ in *nothing classical* but their third circular moment. The green bar is the classical third-harmonic statistic: it sees the difference plainly. The grey bar is the qutrit QMI with harmonics $(1,2)$ — flat on the floor, **blind**, because its $\rho_{AB}$ has no channel for the third moment. The blue bar is the qudit QMI with harmonics $(1,2,3)$: it sees the difference — but it sees exactly what the green classical bar sees, no more. The rose bar is the decisive one: the *same* low-order $(1,2)$ embedding, but run through a genuine entangler (its $\rho_{AB}$ carries more than a bit of entanglement entropy, printed above) — and it stays on the floor, **as blind as the un-entangled qutrit.** Entanglement is real here, and it buys nothing: it cannot conjure access to a moment the embedding never captured.

## 3. The bound, stated — and where it would break

The experiment licenses a precise statement, the sharpened form of `16`'s caveat:

> **Structural bound.** For any *deterministic* embedding of *classical* phase data, the detection power of a quantum-optimal-transport coupling measure is contained in the classical circular moments the embedding captures. A same-order classical statistic matches any QOT measure; **entangling the embedding does not enlarge what it can detect.** There is no quantum-exclusive advantage to be had this way.

This is not pessimism — it is the honest reading of `16` made rigorous, and it is *useful*: it tells you that effort spent hunting a quantum-exclusive win from a cleverer classical-data embedding is effort wasted, and it points at the only place such a win could live.

**When the bound would break (and why it is out of reach here).** Two doors, both outside EEG/MEG:

- **Intrinsically quantum data / a shared quantum resource.** If $A$ and $B$ genuinely shared entanglement, or were probed with incompatible observables, then by Bell/contextuality *no* classical joint distribution would reproduce all the statistics — and a quantum measure could see correlations no classical statistic can, because none exists. Two brains do not share entanglement; the door does not open for hyperscanning.
- **A quantum measurement or channel in the pipeline.** If the embedding passed the signal through a genuine POVM or quantum channel, its output would no longer be a classical moment of the input. Passive analysis of a *recorded* classical signal never does this.

**A note on quantum sensors.** OPM-MEG is built on optically pumped magnetometers — atomic-vapour devices exploiting genuine quantum effects to sense magnetic fields. But the sensor *converts the quantum properties of the atoms into a classical electrical signal* (Brookes et al. 2019): the recorded data is classical, exactly like EEG. A quantum sensor improves the *measurability* side of `17` (higher SNR, better localisation → higher-order structure estimated more reliably) — but it does **not** make the data quantum, and the bound above still holds.

## 4. The right name for it: quantum-inspired

Put `16`, `17` and this notebook together and the programme's identity is clear. Quantum optimal transport on EEG/MEG is a **quantum-inspired** method: it borrows the quantum formalism — density matrices, quantum information measures, optimal transport — as a tool for analysing *classical* data, with no quantum hardware and no quantum-exclusive claim. Its value is **methodological**, and it is real:

- a **unified geometric framework**, in which PLV, higher-order phase coupling, amplitude coupling and multi-channel structure all live as one object $\rho_{AB}$ with one family of distances;
- **composable operations** — partial trace as marginalisation, entropy as uncertainty, transport as comparison;
- a **natural generalisation to networks** of many nodes (multi-subject, multi-region hyperscanning);
- **principled access to higher-order structure** through the embedding (and its weighting, `17`), rather than a statistic hand-picked in advance.

This is an active research direction — density-matrix and quantum-information formalisms are being applied to neural and hyperscanning data (e.g. quantum-inspired interbrain coupling during joint memory tasks; quantum-inspired density-matrix analysis of brain time series). The honest framing is the strong one: not "quantum beats classical", but "a better-organised classical analysis, in a language that unifies and scales." That is a claim that holds — and the bound of this notebook is what keeps it honest.

## Your turn

**Warm-up.** In one or two sentences, explain why the *entangling* qutrit $(1,2)$ embedding stayed blind to the third moment even though its $\rho_{AB}$ carried real entanglement. What does entanglement act on, and what can it *not* create — in terms of which moments the per-sample states contain?

**Go further.** State the structural bound in your own words, then give the single sentence you would add to a methods section to keep a quantum-inspired hyperscanning analysis honest — naming what the method *does* offer (the framework) and what it does *not* (a detection advantage over a matched classical statistic).

**Challenge.** Someone proposes using OPM-MEG "because the quantum sensors will reveal quantum brain correlations." Write the two- or three-sentence correction: distinguish the sensor's quantum mechanism from the (classical) data it outputs, say which side of the work OPM-MEG genuinely helps (point back to `17`), and name what would *actually* be required for the structural bound to break.

## What you built

You found the honest frontier of the whole quantum-inspired program — not by assuming it, but by trying to cross it.

- You stated the **structural bound**: a deterministic embedding of classical phase data builds $\rho_{AB}$ out of classical moments, so a same-order classical statistic matches any QOT measure.
- You ran the **decisive test** — matched on moments 1 and 2, differing only in moment 3 — and watched a too-low-order embedding stay blind *even when made genuinely entangling* (entanglement entropy above a bit, and still no separation). Entanglement is real and buys nothing here.
- You named exactly **when the bound would break** (intrinsically quantum data, or a quantum measurement/channel) and why neither is reachable for EEG or even OPM-MEG — quantum *sensor*, classical *data*.
- You placed the programme correctly: **quantum-inspired**, with a genuine methodological value (unification, composability, networks, principled higher-order access) and no quantum-exclusive overclaim.

Take the measure of this. You did the hardest and most honest thing a researcher can do with a beautiful framework: you found precisely where its advantage is — and is not — and you can now state both without flinching. That is what makes the quantum-inspired claim trustworthy.

## What's next

One frontier remains, and it turns the geometry the other way. In `19_vqe_limitation_de_palma` you meet a result where quantum optimal transport stops being a *measurement* tool and becomes a *proof technique* — bounding what an entire family of quantum algorithms can achieve. A different corner of the map, and a fitting close to the module.

## References

- K. V. Mardia & P. E. Jupp, *Directional Statistics* (Wiley, 2000). DOI:10.1002/9780470316979 — circular moments; the classical content $\rho_{AB}$ is built from.
- M. M. Wilde, *Quantum Information Theory*, 2nd ed. (Cambridge University Press, 2017). DOI:10.1017/9781316809976 — quantum mutual information, the coupling measure tested here.
- M. J. Brookes et al., "Magnetoencephalography with optically pumped magnetometers (OPM-MEG): the next generation of functional neuroimaging", *Trends in Neurosciences* **45**, 621–634 (2022). DOI:10.1016/j.tins.2022.05.008 — OPM-MEG; the sensor is quantum, the recorded signal is classical.
- E. Boto et al. / R. M. Hill, J. Osborne et al., "Optically pumped magnetometers: from quantum origins to multi-channel magnetoencephalography", *NeuroImage* **199**, 598–608 (2019). DOI:10.1016/j.neuroimage.2019.05.063 — the magnetometer converts the atoms' quantum properties into a classical electrical signal.
- D. Trevisan, "Optimal transport methods for quantum systems", arXiv:2202.02091 (2022). DOI:10.48550/arXiv.2202.02091 — the QOT coupling measures used throughout.

> Note on related quantum-inspired neuroscience: density-matrix and quantum-information formalisms applied to neural / hyperscanning data are an active, distinct literature; cite the specific paper you build on when positioning this work, and verify each reference before use.

**Previous:** `notebooks/04_QuantumOptimalTransport/17_embedding_design_amplifies_the_effect.ipynb` · **Next:** `notebooks/04_QuantumOptimalTransport/19_vqe_limitation_de_palma.ipynb`